# BiLSTM + attention (starter) — Jigsaw toxic comments

**Preprocessing:** `preprocess_for_bilstm` (word indices + sequence lengths).

**Run modes** (set flags in the next code cell): **quick** smoke test (`QUICK_ITERATION = True`), **progress report** (`QUICK_ITERATION = False`, `PROGRESS_REPORT_MODE = True`), or **full** (both `False`).

**Metrics:** Same proposal bundle as other starter notebooks.

In [13]:
#1) PATHFINDER
from pathlib import Path
import sys

# Search up to 3 levels up to find the repo root
ROOT = Path.cwd().resolve()
for p in [ROOT] + list(ROOT.parents)[:3]:
    if (p / "preprocessing" / "text_preprocessing.py").exists():
        ROOT = p
        break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Add notebooks folder for metrics_helpers.py
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

print(f"Project Root identified at: {ROOT}")

Project Root identified at: C:\Users\Prana\Desktop\CurSem\258_DL\Project\cmpe258-2026Fall-ToxicCommentDetection


In [14]:
#2-1) Quickset
QuickSetEpochs = 8
QuickSetSamples = 75_000 
QuickSetBatchSize = 128
QuickSetMaxLen = 100
QuickSetVocab = 20_000

In [15]:
#2-2) Initialize)
import time
import torch
import numpy as np
import pandas as pd
import os
import torch.nn as nn
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset

from preprocessing.text_preprocessing import LABEL_COLUMNS
from metrics_helpers import multilabel_evaluation_report, per_label_confusion_matrices, torch_parameter_count

# Progression/QuickSet Configuration
QuickSetEpochs = 15
QuickSetSamples = 100_000 
QuickSetBatchSize = 128
QuickSetMaxLen = 100
QuickSetVocab = 25_000
DataIncrements = [10000, 30000,50000,70000,90000,110000]

def pick_device() -> torch.device:
    if torch.cuda.is_available(): return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = pick_device()
print("Device:", DEVICE)
RNG = np.random.default_rng(42)
torch.manual_seed(42)

from preprocessing.text_preprocessing import preprocess_for_bilstm

QUICK_ITERATION = False
PROGRESS_REPORT_MODE = True
LR = 1e-3

PROGRESS_MAX_TRAIN_SAMPLES = QuickSetSamples # 10_000
PROGRESS_MAX_VAL_SAMPLES = 5_000 # 2_000
PROGRESS_MAX_LEN = QuickSetMaxLen # 128
PROGRESS_BATCH_SIZE = QuickSetBatchSize # 64
PROGRESS_EPOCHS = QuickSetEpochs # 2
PROGRESS_MAX_VOCAB = QuickSetVocab # 10_000
PROGRESS_MIN_FREQ = 2
PROGRESS_HIDDEN = 128

# Set active config based on mode
if PROGRESS_REPORT_MODE:
    run_mode = "progress report"
    MAX_LEN = PROGRESS_MAX_LEN
    BATCH_SIZE = PROGRESS_BATCH_SIZE
    EPOCHS = PROGRESS_EPOCHS
    MAX_VOCAB = PROGRESS_MAX_VOCAB
    MIN_FREQ = PROGRESS_MIN_FREQ
    HIDDEN = PROGRESS_HIDDEN

Device: cpu


In [16]:
#3) MODEL ARCHITECTURE
class AttentionBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden, num_labels, padding_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.spatial_dropout = nn.Dropout1d(0.2)
        self.lstm = nn.LSTM(embed_dim, hidden, batch_first=True, bidirectional=True)
        self.layer_norm = nn.LayerNorm(hidden * 2)
        self.attn_net = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1)
        )
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(hidden * 2, num_labels)

    def forward(self, x, lengths):
        emb = self.embedding(x).permute(0, 2, 1)
        emb = self.spatial_dropout(emb).permute(0, 2, 1)
        
        lens = lengths.clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(emb, lens, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        
        out = self.layer_norm(out)
        scores = self.attn_net(out).squeeze(-1)
        mask = torch.arange(out.size(1), device=x.device).unsqueeze(0) < lengths.unsqueeze(1)
        scores = scores.masked_fill(~mask, -1e9)
        
        w = torch.softmax(scores, dim=1).unsqueeze(-1)
        ctx = (out * w).sum(dim=1)
        return self.fc(self.dropout(ctx))

def predict_logits(model, X_np, len_np, batch_size=512):
    model.eval()
    all_logits = []
    x_tensor = torch.tensor(X_np, dtype=torch.long)
    len_tensor = torch.tensor(len_np, dtype=torch.long)
    with torch.no_grad():
        for i in range(0, len(x_tensor), batch_size):
            logits = model(x_tensor[i:i+batch_size].to(DEVICE), len_tensor[i:i+batch_size].to(DEVICE))
            all_logits.append(logits.cpu().numpy())
    return np.concatenate(all_logits, axis=0)

def find_best_thresholds_brute_force(y_true, y_probs, labels):
    best_thresholds = []
    from sklearn.metrics import f1_score
    for i, label in enumerate(labels):
        y_t, y_p = y_true[:, i], y_probs[:, i]
        cur_best_f1, cur_best_t = -1.0, 0.5
        for t in np.arange(0.1, 1.0, 0.1):
            s = f1_score(y_t, (y_p >= t).astype(int), zero_division=0)
            if s > cur_best_f1: cur_best_f1, cur_best_t = s, t
        for t in np.arange(max(0.01, cur_best_t-0.1), min(0.99, cur_best_t+0.1), 0.01):
            s = f1_score(y_t, (y_p >= t).astype(int), zero_division=0)
            if s > cur_best_f1: cur_best_f1, cur_best_t = s, t
        best_thresholds.append(cur_best_t)
    return np.array(best_thresholds)

In [ ]:
#4) TEST EVERYTHING
from sklearn.metrics import f1_score, precision_score, recall_score

# Configuration for stopping sensitivity
min_delta = 0.005  # Minimum improvement in F1 to reset patience
patience_limit = 3 # Stop after 3 epochs of no significant improvement

csv_file = "bilstm_data.csv"
if not os.path.exists(csv_file):
    pd.DataFrame(columns=["Train_Data_Size", "Best Epoch", "Label", "Precision", "Recall", "F1", "Roc_Auc"]).to_csv(csv_file, index=False)

for current_size in DataIncrements:
    print(f"\n{'='*40}\nSTAGE: {current_size} SAMPLES\n{'='*40}")
    
    data = preprocess_for_bilstm(
        max_len=PROGRESS_MAX_LEN, validation_fraction=0.1, random_state=42,
        min_freq=PROGRESS_MIN_FREQ, max_vocab=PROGRESS_MAX_VOCAB, 
        max_train_samples=current_size, max_val_samples=PROGRESS_MAX_VAL_SAMPLES
    )

    pos_weights = (torch.tensor(len(data.y_train) - data.y_train.sum(axis=0)) / (torch.tensor(data.y_train.sum(axis=0)) + 1e-5)).to(DEVICE)
    model = AttentionBiLSTM(len(data.vocab), 100, HIDDEN, len(LABEL_COLUMNS)).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

    train_loader = DataLoader(TensorDataset(torch.tensor(data.X_train, dtype=torch.long), torch.tensor(data.length_train, dtype=torch.long), torch.tensor(data.y_train, dtype=torch.float32)), batch_size=PROGRESS_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.tensor(data.X_val, dtype=torch.long), torch.tensor(data.length_val, dtype=torch.long), torch.tensor(data.y_val, dtype=torch.float32)), batch_size=PROGRESS_BATCH_SIZE)

    best_f1_macro = -1.0
    best_epoch = 0
    best_model_state = None
    patience_counter = 0

    for epoch in range(PROGRESS_EPOCHS):
        model.train()
        for xb, lb, yb in train_loader:
            optimizer.zero_grad()
            loss_fn(model(xb.to(DEVICE), lb.to(DEVICE)), yb.to(DEVICE)).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
        
        # Validation Loss
        model.eval()
        v_loss = 0
        with torch.no_grad():
            for xv, lv, yv in val_loader:
                v_loss += loss_fn(model(xv.to(DEVICE), lv.to(DEVICE)), yv.to(DEVICE)).item()
        v_loss /= len(val_loader)

        # Brute Force Threshold Search for every epoch
        epoch_probs = torch.sigmoid(torch.from_numpy(predict_logits(model, data.X_val, data.length_val))).numpy()
        best_thresholds_this_epoch = find_best_thresholds_brute_force(data.y_val, epoch_probs, LABEL_COLUMNS)
        epoch_preds = (epoch_probs >= best_thresholds_this_epoch).astype(int)
        
        cur_f1 = f1_score(data.y_val, epoch_preds, average='macro', zero_division=0)
        cur_prec = precision_score(data.y_val, epoch_preds, average='macro', zero_division=0)
        cur_recall = recall_score(data.y_val, epoch_preds, average='macro', zero_division=0)
        
        print(f"Epoch {epoch+1:2d} -> Val Loss: {v_loss:.4f}, F1: {cur_f1:.4f} ({cur_prec:.4f}, {cur_recall:.4f})")
        
        scheduler.step()

        # Early Stopping Logic with Min Delta
        # This prevents the code from running forever on tiny 0.0001 gains
        if cur_f1 > (best_f1_macro + min_delta):
            best_f1_macro = cur_f1
            best_epoch = epoch + 1
            best_model_state = model.state_dict().copy()
            patience_counter = 0 # Reset because we had a SIGNIFICANT improvement
        else:
            patience_counter += 1
            if patience_counter >= patience_limit: 
                print(f"Early stopping triggered: F1 hasn't improved by {min_delta} for {patience_limit} epochs.")
                break

    # Save the BESTEST version found
    if best_model_state:
        model.load_state_dict(best_model_state)
        torch.save(model.state_dict(), f"best_model_{current_size}.pt")
        
        # Log Final Results
        final_probs = torch.sigmoid(torch.from_numpy(predict_logits(model, data.X_val, data.length_val))).numpy()
        final_thresholds = find_best_thresholds_brute_force(data.y_val, final_probs, LABEL_COLUMNS)
        res_df, _ = multilabel_evaluation_report(data.y_val, (final_probs >= final_thresholds).astype(int), final_probs, LABEL_COLUMNS)
        
        res_df['Train_Data_Size'], res_df['Best Epoch'] = current_size, best_epoch
        res_df[["Train_Data_Size", "Best Epoch", "label", "precision", "recall", "f1", "roc_auc"]].to_csv(csv_file, mode='a', header=False, index=False)


STAGE: 10000 SAMPLES


In [6]:
#5) DISPLAY
final_results = pd.read_csv(csv_file)
display(final_results.tail(len(LABEL_COLUMNS)))
print(f"Progression training complete. Models and {csv_file} saved.")

,Train_Data_Size,Best Epoch,Label,Precision,Recall,F1,Roc_Auc
36,140000,3,toxic,0.758454,0.666667,0.709605,0.956778
37,140000,3,severe_toxic,0.357143,0.555556,0.434783,0.986278
38,140000,3,obscene,0.805556,0.664122,0.728033,0.966813
39,140000,3,threat,0.250000,0.777778,0.378378,0.996082
40,140000,3,insult,0.648980,0.657025,0.652977,0.958531
41,140000,3,identity_hate,0.213483,0.475000,0.294574,0.952737


Progression training complete. Models and bilstm_data.csv saved.


## Notes

- Increase `EPOCHS` and add validation-based **early stopping** for the final report.
- Tune **class weights** in `BCEWithLogitsLoss` (`pos_weight`) for rare heads (`threat`, `identity_hate`).